# ksmart-image-api — Colab Setup (Drive + git + OpenCode)
Chạy từng cell theo thứ tự. Model được lưu trên Google Drive để không tải lại mỗi session.
**Lưu ý:** Colab free ngắt sau ~12h / 90p không thao tác. Drive giữ model + repo, nhưng process phải chạy lại.

In [ ]:
# === CELL 1: Mount Drive + clone/pull repo ===
from google.colab import drive
import os
drive.mount('/content/drive')

WORK = '/content/drive/MyDrive/ksmart-image-api'
os.makedirs(os.path.dirname(WORK), exist_ok=True)
if not os.path.exists(WORK):
    !git clone --recurse-submodules https://github.com/hoainamgo/ksmart-image-api "{WORK}"
else:
    %cd "{WORK}"
    !git pull
%cd "{WORK}"
!echo "repo ready: $(pwd)"

In [ ]:
# === CELL 2: Install deps + prepare model dir on Drive ===
!pip install -q -r requirements.txt
!pip install -q -r ComfyUI/requirements.txt

MODELDIR = '/content/drive/MyDrive/ksmart-models'
!mkdir -p "{MODELDIR}/diffusion_models" "{MODELDIR}/loras" "{MODELDIR}/text_encoders/split_files/text_encoders" "{MODELDIR}/vae/split_files/vae"

# Link model dir into repo (so run_workflow finds klein + lora)
!ln -sfn "{MODELDIR}" models
!ls -la models
!echo '--- model dir contents (should have klein + lora) ---'
!ls -lh "{MODELDIR}/diffusion_models" "{MODELDIR}/loras" 2>/dev/null

In [ ]:
# === CELL 2b: Download models ONCE (if not on Drive yet) ===
# Small Klein 4B fp8 (~3.8GB) fits T4. LoRA anatomy fixer (~20MB).
MODELDIR = '/content/drive/MyDrive/ksmart-models'
!wget -c -P "{MODELDIR}/diffusion_models" "https://huggingface.co/black-forest-labs/FLUX.2-klein-4b-fp8/resolve/main/flux-2-klein-4b-fp8.safetensors"
!wget -c -P "{MODELDIR}/loras" "https://civitai.com/api/download/models/2615554?fileId=2502989" -O "{MODELDIR}/loras/klein_anatomy_fixer.safetensors"
!echo done

In [ ]:
# === CELL 3: Start API server (auto-launches ComfyUI + worker) ===
import os, subprocess
os.environ['API_KEY'] = 'ksmart_3xS33jgnnArmLawramxsByHnBmgyQ1w4Z96SdkcLf'
print('starting server + worker...')
subprocess.Popen(['setsid','python','server.py','>','api_server.log','2>&1','&'], cwd='/content/drive/MyDrive/ksmart-image-api')
subprocess.Popen(['setsid','python','worker.py','>','worker.log','2>&1','&'], cwd='/content/drive/MyDrive/ksmart-image-api')
!sleep 20
!curl -s -o /dev/null -w "local /queue: %{http_code}\n" "http://127.0.0.1:8000/queue?api_key=ksmart_3xS33jgnnArmLawramxsByHnBmgyQ1w4Z96SdkcLf"

In [ ]:
# === CELL 4: Launch OpenCode (web UI on port 4096) ===
!pip install -q uv
import subprocess, os
proc = subprocess.Popen(
    ['uvx', 'opencode-ai', '--address', '0.0.0.0:4096'],
    cwd='/content/drive/MyDrive/ksmart-image-api'
)
print('opencode pid', proc.pid)
# Then: Colab menu ▸ Code Snippets ▸ forward localhost:4096  (or run next cell)

In [ ]:
# === CELL 5: Forward OpenCode port to browser ===
from google.colab.output import serve_kernel_port
serve_kernel_port(4096)
print('Open the forwarded localhost:4096 link above to use OpenCode on this repo.')

In [ ]:
# === CELL 6: Quick test prompt (optional) ===
import requests
H = {'X-API-Key':'ksmart_3xS33jgnnArmLawramxsByHnBmgyQ1w4Z96SdkcLf'}
r = requests.post('http://127.0.0.1:8000/prompt', headers=H, json={
    'client_id':'colab_test','prompt':{'prompt':'a red fox in snow, illustration'},
    'settings':{'seed':42,'lora_strength':3.0}})
pid = r.json()['prompt_id']
print('job', pid)